# 02 - Feature Engineering

Goal: build the early-window feature table (day-1/day-2 cutoff only) for
each of the 607 events, join it against the `severity_class` labels from
`01_data_ingestion.ipynb`, and produce a single train-ready DataFrame for
the GBM (XGBoost) and EBM (InterpretML) baselines in Week 2.

Everything here reads ONLY from the early cutoff window -- reaching into
later days would leak the label, since severity_class is derived from the
full trajectory.

In [3]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from flashpoint.db import get_connection

DB_PATH = Path("../data/flashpoint.duckdb")
CUTOFF_DAY = 2  # use the first 2 days as the "early" window

con = get_connection(DB_PATH)
events_df = con.execute("SELECT * FROM events").df()
outcomes_df = con.execute("SELECT * FROM event_outcomes").df()
print(f"{len(events_df)} events, {len(outcomes_df)} outcomes")

607 events, 607 outcomes


## Step 1: Filter out events too short for the cutoff

An event needs at least `CUTOFF_DAY` days of observations to compute early
features at all -- check for any that don't, per the note flagged at the
end of `01_data_ingestion.ipynb`.

In [4]:
too_short = events_df[events_df['n_days'] < CUTOFF_DAY]
print(f"{len(too_short)} events have fewer than {CUTOFF_DAY} days -- these get dropped")

usable_events = events_df[events_df['n_days'] >= CUTOFF_DAY].copy()
print(f"{len(usable_events)} usable events remain")

0 events have fewer than 2 days -- these get dropped
607 usable events remain


## Step 2: Compute early-window features per event

In [5]:
import pandas as pd
from tqdm import tqdm
from flashpoint.data_access import HDF5Event, read_event_window
from flashpoint.features import early_window_stats

feature_rows = []
for row in tqdm(usable_events.itertuples(), total=len(usable_events)):
    event = HDF5Event(
        event_id=row.event_id, year=row.year, hdf5_path=Path(row.hdf5_path),
        n_days=row.n_days, img_dates=[], lnglat=(row.centroid_lon, row.centroid_lat),
    )
    early_stack = read_event_window(event, 0, CUTOFF_DAY)
    stats = early_window_stats(early_stack)
    stats["event_id"] = row.event_id
    feature_rows.append(stats)

features_df = pd.DataFrame(feature_rows)
features_df.head()

100%|██████████| 607/607 [00:05<00:00, 121.14it/s]


,fire_extent_ha,wind_speed_mean,wind_speed_max,wind_direction_sin_mean,wind_direction_cos_mean,max_temp_max,min_temp_min,humidity_min,pdsi_mean,erc_mean,event_id
0,0.0,0.895985,2.4,-0.354610,-0.385744,294.899994,274.100006,0.00178,-1.642182,52.257618,fire_21458798
1,0.0,4.423371,9.5,-0.023865,-0.802851,292.399994,259.299988,0.00131,-2.819414,61.060646,fire_21458801
2,0.0,1.746247,4.0,0.027330,0.047859,295.200012,272.100006,0.00349,-1.862331,26.161974,fire_21458806
3,112.5,1.051939,2.6,-0.341254,-0.491015,299.399994,271.799988,0.00073,-1.110331,64.648163,fire_21458836
4,0.0,4.598395,9.1,-0.122574,-0.069806,287.000000,250.300003,0.00114,-2.819055,49.521889,fire_21458848


## Step 3: Write early_features to DuckDB, then join with labels

In [6]:
insert_rows = [
    (
        r["event_id"], CUTOFF_DAY, r["fire_extent_ha"], r["wind_speed_mean"],
        r["wind_speed_max"], r["wind_direction_sin_mean"], r["wind_direction_cos_mean"],
        r["max_temp_max"], r["min_temp_min"], r["humidity_min"], r["pdsi_mean"], r["erc_mean"],
    )
    for r in feature_rows
]
con.executemany(
    "INSERT OR REPLACE INTO early_features VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    insert_rows,
)
con.commit()

train_df = con.execute("""
    SELECT e.year, f.*, o.severity_class
    FROM early_features f
    JOIN events e ON e.event_id = f.event_id
    JOIN event_outcomes o ON o.event_id = f.event_id
""").df()
print(train_df.shape)
train_df.head()

(607, 14)


,year,event_id,cutoff_day,fire_extent_ha,wind_speed_mean,wind_speed_max,wind_direction_sin_mean,wind_direction_cos_mean,max_temp_max,min_temp_min,humidity_min,pdsi_mean,erc_mean,severity_class
0,2018,fire_21458798,2,0.0,0.895985,2.4,-0.354610,-0.385744,294.899994,274.100006,0.00178,-1.642182,52.257618,0
1,2018,fire_21458801,2,0.0,4.423371,9.5,-0.023865,-0.802851,292.399994,259.299988,0.00131,-2.819414,61.060646,0
2,2018,fire_21458806,2,0.0,1.746247,4.0,0.027330,0.047859,295.200012,272.100006,0.00349,-1.862331,26.161974,1
3,2018,fire_21458836,2,112.5,1.051939,2.6,-0.341254,-0.491015,299.399994,271.799988,0.00073,-1.110331,64.648163,0
4,2018,fire_21458848,2,0.0,4.598395,9.1,-0.122574,-0.069806,287.000000,250.300003,0.00114,-2.819055,49.521889,0


## Step 4: Year-based train/test split (not random)

Per the paper's own recommendation: yearly distributions vary a lot (2019
is a known outlier -- fewer, smaller fires), so a random shuffle risks an
unrepresentative split. Hold out one full year as the test set instead --
e.g. train on 2018/2020/2021, test on 2019, or run this across a few
held-out years to sanity check stability before committing to one split.